## backend > beamline.py

In [41]:
from sympy import symbols, Matrix
import sympy as sp
import numpy as np
from scipy import interpolate
from scipy import optimize
import math
import torch

In [54]:
# from ..backend.beamline import *
import os, sys
from pprint import pprint
pprint(sys.path)

current_dir = os.path.abspath(os.getcwd())

project_root = os.path.abspath(os.path.join(current_dir, '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
pprint(sys.path)
from backend.beamline import *

['C:\\Users\\yi_lu\\my_files\\research\\FELsim',
 'C:\\Users\\yi_lu\\programming_IDE\\PyCharm_2025.2.1\\plugins\\python-ce\\helpers\\pydev',
 'C:\\Users\\yi_lu\\programming_IDE\\PyCharm_2025.2.1\\plugins\\python-ce\\helpers\\jupyter_debug',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\python311.zip',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\DLLs',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\Lib',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim',
 '',
 'C:\\Users\\yi_lu\\AppData\\Roaming\\Python\\Python311\\site-packages',
 'C:\\Users\\yi_lu\\programming_lang\\Anaconda\\envs\\felsim\\Lib\\site-packages',
 'C:\\Users\\yi_lu\\my_files\\research\\FELsim']
['C:\\Users\\yi_lu\\my_files\\research\\FELsim',
 'C:\\Users\\yi_lu\\programming_IDE\\PyCharm_2025.2.1\\plugins\\python-ce\\helpers\\pydev',
 'C:\\Users\\yi_lu\\programming_IDE\\PyCharm_2025.2.1\\plugins\\python-ce\\helpers\\jupyter_debug',
 'C:\\Users\\yi_lu\\programmin

## lattice > useMatrice()

In [5]:
# before
def useMatrice(val, **kwargs):
    ''''
    Simulates the movement of given particles through its child segment by
    applying the segment's transfer matrix with numeric parameters.

    Parameters
    ----------
    val : np.ndarray
        A 2D NumPy array representing the particle states. Each row is a particle,
        and columns correspond to phase space coordinates (e.g., [x, x', y, y', z, z']).
    **kwargs : dict
        Other segment-specific numeric parameters (e.g., `length`, `current`)
        that might override the segment's default properties for this specific simulation.

    Returns
    -------
    list
        A 2D list where each inner list represents the transformed state of a particle
        after passing through the segment.

    '''
    np.random.seed(42)
    mat = np.random.randn(6,6)
    npMat = np.array(mat).astype(np.float64)

    newMatrix = []
    for array in val:
        tempArray = np.matmul(npMat, array)
        newMatrix.append(tempArray.tolist())
    return newMatrix #  return 2d list

In [20]:
N=1000
torch.manual_seed(42)
val=np.random.randn(N,6)


matrix_1=useMatrice(val)
print(matrix_1)
# print(type(matrix_1[0]))

[[-0.3987655063485043, -0.8657502604857141, 5.286164518388932, 0.3926060221460539, 0.7780819143953008, -3.5476162302993233], [-2.209429472842908, -0.38900630172552325, 4.30194872185634, -3.0114479678636616, 1.429204519505724, -1.3762754749534576], [0.053146790680521305, -1.1426294346279429, 3.9916666177875566, 1.8226741702597362, -0.6721506643492992, -4.373026573044818], [-0.9371196092660791, 1.96123548920314, 0.06015920796151458, -4.779144576604804, -0.09172487372770044, 0.5245060115951923], [-3.2585630784487885, -2.0369974055291338, 2.423467930082295, -2.531546892930399, 0.1805912089439824, 0.2371854608936136], [-1.3675635622018079, -0.7471548709035392, -2.0812218641617606, -2.8431325675344827, -1.1737992579492493, 0.9991821337067337], [-0.6295001794138625, 2.9189370908941408, 0.8723029298628754, -5.770525101348155, 3.552089072758851, 2.5223774500328653], [0.4972951817433184, -1.5270426677655073, 2.3433728815423334, 3.1593446382523727, -0.6199160147827307, -2.263139383546614], [0.765

In [32]:
# after
def useMatrice(val_tensor, **kwargs):
    '''
    [Refactored for PyTorch]
    Simulates the movement of given particles through its child segment by
    applying the segment's transfer matrix via batched tensor operations.
    '''

    # actual
    # mat = self.getSymbolicMatrice(numeric = True, **kwargs)
    # 1. get the matrix

    np.random.seed(42)
    mat = np.random.randn(6,6)
    mat = torch.Tensor(mat)

    # transform into torch tensor
    mat_tensor = torch.tensor(np.array(mat).astype(np.float64),
                              dtype=val_tensor.dtype,
                              device=val_tensor.device)

    # parallelized matrix multiplication
    new_val_tensor = torch.matmul(val_tensor, mat_tensor.T)
    new_val_tensor = new_val_tensor.cpu().numpy().tolist()
    return new_val_tensor

In [43]:
torch.manual_seed(42)
val_tensor=torch.from_numpy(val.copy())
matrix_2=useMatrice(val_tensor)
print(matrix_2)
print(type(matrix_2[0]))

[[-0.39876556260295865, -0.8657502443670494, 5.2861646263713284, 0.3926059907476306, 0.7780819238119346, -3.5476163070819204], [-2.2094294755330117, -0.3890063035934634, 4.301948786775742, -3.011447970468049, 1.4292045280382608, -1.376275551997347], [0.05314681479945535, -1.1426294209038204, 3.9916665833439673, 1.8226741481174291, -0.672150680786678, -4.373026706121066], [-0.9371196208677297, 1.9612354519435995, 0.06015925914819535, -4.779144557573238, -0.09172489413904024, 0.5245060156234344], [-3.2585630415346696, -2.036997371897942, 2.4234680036014895, -2.531546895423494, 0.18059122229142927, 0.23718539210901826], [-1.3675635100337236, -0.7471548552482463, -2.081221879426405, -2.843132546944136, -1.1737992612591697, 0.9991821272673582], [-0.6295002960402101, 2.9189370397122505, 0.8723030873116127, -5.770525082145516, 3.552089104343016, 2.522377570250785], [0.4972951484436921, -1.5270426332240794, 2.3433729468539606, 3.1593446039086555, -0.6199160096453896, -2.2631394230117112], [0.7

C:\Users\yi_lu\AppData\Local\Temp\ipykernel_2776\57284834.py:18: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  mat_tensor = torch.tensor(np.array(mat).astype(np.float64),


In [40]:
print(matrix_1[1])
print(matrix_2[1])
print(np.allclose(matrix_1[1],matrix_2[1]))

[-2.209429472842908, -0.38900630172552325, 4.30194872185634, -3.0114479678636616, 1.429204519505724, -1.3762754749534576]
[-2.2094294755330117, -0.3890063035934634, 4.301948786775742, -3.011447970468049, 1.4292045280382608, -1.376275551997347]
True


## driftLattice(lattice) > getSymbolicMatrice
same for qpfLattice, qpdLattice, dipole, dipole_wedge, Beamline

In [51]:
# before
class driftLattice(lattice):
    ...
    def getSymbolicMatrice(self, numeric = False, length = None):
        ...
        return mat

Matrix([[1, 1, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0], [0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 1]])
tensor([1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0.,
        0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1.],
       dtype=torch.float64)
object



In [ ]:
# after
class driftLattice(lattice):
    ...
    def getSymbolicMatrice(self, numeric = False, length = None):
        ...
        return torch.tensor(mat,dtype=torch.float32)
# same for qpfLattice, qpdLattice, dipole, dipole_wedge, Beamline